In [1]:
!pip install ultralytics opencv-python pyyaml -q

from google.colab import drive
drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/CSCI435_Buzzlytics_data")

assert DATA_ROOT.exists(), f"Path not found: {DATA_ROOT}"

print("Root exists:", DATA_ROOT)

# Show relevant gt files
gt_files = list(DATA_ROOT.rglob("gt_one.csv")) + list(DATA_ROOT.rglob("gt.csv"))

print("Found GT files:")
for p in gt_files:
    print(p)

Root exists: /content/drive/MyDrive/CSCI435_Buzzlytics_data
Found GT files:
/content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa/train/gt_one.csv
/content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa/test/gt_one.csv
/content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa/val/gt_one.csv


In [3]:
def find_varroa_source(split_keyword):
    """
    Finds the folder containing gt_one.csv/gt.csv for train_varroa, val_varroa, or test_varroa.
    Example return:
    /content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa/train
    """
    candidates = []

    for gt in gt_files:
        path_text = str(gt).lower()
        if split_keyword.lower() in path_text:
            candidates.append(gt.parent)

    if not candidates:
        raise FileNotFoundError(f"Could not find source for: {split_keyword}")

    # Prefer folders that directly contain videos/ and labels/
    candidates = sorted(
        candidates,
        key=lambda p: int((p / "videos").exists()) + int((p / "labels").exists()),
        reverse=True
    )

    return candidates[0]


TRAIN_SRC = find_varroa_source("train_varroa")
VAL_SRC   = find_varroa_source("val_varroa")
TEST_SRC  = find_varroa_source("test_varroa")

print("TRAIN_SRC:", TRAIN_SRC)
print("VAL_SRC:", VAL_SRC)
print("TEST_SRC:", TEST_SRC)

TRAIN_SRC: /content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa/train
VAL_SRC: /content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa/val
TEST_SRC: /content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa/test


In [5]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/CSCI435_Buzzlytics_data")
assert DATA_ROOT.exists(), DATA_ROOT

gt_files = list(DATA_ROOT.rglob("gt_one.csv")) + list(DATA_ROOT.rglob("gt.csv"))

print("GT files found:")
for p in gt_files:
    print(p)

def find_varroa_source(split_keyword):
    candidates = []
    for gt in gt_files:
        if split_keyword.lower() in str(gt).lower():
            candidates.append(gt.parent)

    if not candidates:
        raise FileNotFoundError(f"Could not find {split_keyword}")

    candidates = sorted(
        candidates,
        key=lambda p: int((p / "videos").exists()) + int((p / "labels").exists()),
        reverse=True
    )
    return candidates[0]

TRAIN_SRC_DRIVE = find_varroa_source("train_varroa")
VAL_SRC_DRIVE   = find_varroa_source("val_varroa")
TEST_SRC_DRIVE  = find_varroa_source("test_varroa")

print("TRAIN:", TRAIN_SRC_DRIVE)
print("VAL:", VAL_SRC_DRIVE)
print("TEST:", TEST_SRC_DRIVE)

GT files found:
/content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa/train/gt_one.csv
/content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa/test/gt_one.csv
/content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa/val/gt_one.csv
TRAIN: /content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa/train
VAL: /content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa/val
TEST: /content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa/test


In [7]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/CSCI435_Buzzlytics_data")

zip_files = list(DATA_ROOT.rglob("*.zip"))

print("Zip files found:")
for z in zip_files:
    print(z)

Zip files found:
/content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa-20260628T085827Z-3-001.zip
/content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa-20260628T085824Z-3-001.zip
/content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa-20260628T085825Z-3-001.zip


In [8]:
from pathlib import Path
import zipfile
import shutil

DATA_ROOT = Path("/content/drive/MyDrive/CSCI435_Buzzlytics_data")
LOCAL_RAW = Path("/content/local_varroa_raw")

shutil.rmtree(LOCAL_RAW, ignore_errors=True)
LOCAL_RAW.mkdir(parents=True, exist_ok=True)

zip_files = list(DATA_ROOT.rglob("*.zip"))

for z in zip_files:
    if "varroa" in z.name.lower():
        print("Extracting:", z)
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(LOCAL_RAW)

print("Done extracting to:", LOCAL_RAW)

Extracting: /content/drive/MyDrive/CSCI435_Buzzlytics_data/train_varroa-20260628T085827Z-3-001.zip
Extracting: /content/drive/MyDrive/CSCI435_Buzzlytics_data/val_varroa-20260628T085824Z-3-001.zip
Extracting: /content/drive/MyDrive/CSCI435_Buzzlytics_data/test_varroa-20260628T085825Z-3-001.zip
Done extracting to: /content/local_varroa_raw


In [10]:
from pathlib import Path
import os
import shutil
import yaml
import hashlib
from PIL import Image

TRAIN_SRC = Path("/content/local_varroa_raw/train_varroa/train")
VAL_SRC   = Path("/content/local_varroa_raw/val_varroa/val")
TEST_SRC  = Path("/content/local_varroa_raw/test_varroa/test")

OUT = Path("/content/varroa_det")
shutil.rmtree(OUT, ignore_errors=True)

for split in ["train", "valid", "test"]:
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def safe_stem(text):
    flat = text.replace("\\", "/").replace("/", "__")
    flat = "".join(c if (c.isalnum() or c in "._-") else "_" for c in flat)
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()[:8]
    return f"{flat[:90]}_{digest}"

def index_images(source):
    by_rel = {}
    by_base = {}
    root = source / "videos"

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rel_from_source = str(p.relative_to(source)).replace("\\", "/")
            rel_from_videos = str(p.relative_to(root)).replace("\\", "/")
            by_rel[rel_from_source] = p
            by_rel[rel_from_videos] = p
            by_base.setdefault(p.name, p)

    return by_rel, by_base

def resolve_image(raw, source, by_rel, by_base):
    norm = raw.replace("\\", "/").lstrip("./")

    candidates = [
        by_rel.get(norm),
        by_rel.get(f"videos/{norm}"),
        by_base.get(Path(norm).name),
        source / norm,
        source / "videos" / norm,
    ]

    return next((p for p in candidates if p is not None and p.is_file()), None)

def boxes_to_yolo(coords, img_w, img_h):
    lines = []

    for i in range(0, len(coords), 4):
        if i + 3 >= len(coords):
            continue

        x1, y1, x2, y2 = map(float, coords[i:i+4])

        x1 = max(0.0, min(float(img_w), x1))
        x2 = max(0.0, min(float(img_w), x2))
        y1 = max(0.0, min(float(img_h), y1))
        y2 = max(0.0, min(float(img_h), y2))

        if x2 <= x1 or y2 <= y1:
            continue

        cx = ((x1 + x2) / 2.0) / img_w
        cy = ((y1 + y2) / 2.0) / img_h
        bw = (x2 - x1) / img_w
        bh = (y2 - y1) / img_h

        lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

    return lines

def convert_split(source, split):
    csv_path = source / "gt_one.csv"
    if not csv_path.exists():
        csv_path = source / "gt.csv"

    by_rel, by_base = index_images(source)

    written = 0
    positives = 0
    boxes = 0
    missing = 0

    rows = csv_path.read_text(encoding="utf-8").splitlines()
    print(f"\nConverting {split}: {len(rows)} rows")

    for line in rows:
        parts = line.split()
        if len(parts) < 2:
            continue

        raw_img = parts[0]
        mite_count = int(float(parts[1]))
        coords = [float(x) for x in parts[2:]]

        img_path = resolve_image(raw_img, source, by_rel, by_base)
        if img_path is None:
            missing += 1
            continue

        try:
            with Image.open(img_path) as im:
                img_w, img_h = im.size
        except Exception:
            missing += 1
            continue

        label_lines = boxes_to_yolo(coords, img_w, img_h) if mite_count > 0 else []

        stem = safe_stem(raw_img)
        dst_img = OUT / split / "images" / f"{stem}{img_path.suffix.lower()}"
        dst_lbl = OUT / split / "labels" / f"{stem}.txt"

        if dst_img.exists() or dst_img.is_symlink():
            dst_img.unlink()

        # Fast: symlink image instead of copying again
        os.symlink(img_path, dst_img)
        dst_lbl.write_text("".join(label_lines), encoding="utf-8")

        written += 1
        if label_lines:
            positives += 1
            boxes += len(label_lines)

        if written % 1000 == 0:
            print(f"{split}: written={written}, positives={positives}, boxes={boxes}, missing={missing}")

    print(f"{split} DONE: images={written}, positives={positives}, boxes={boxes}, missing={missing}")

convert_split(TRAIN_SRC, "train")
convert_split(VAL_SRC, "valid")
convert_split(TEST_SRC, "test")

data_yaml = {
    "path": str(OUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": {0: "mite"},
}

with open(OUT / "varroa_mite.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\nYOLO dataset ready:", OUT)
print("YAML:", OUT / "varroa_mite.yaml")


Converting train: 8225 rows
train: written=1000, positives=413, boxes=414, missing=0
train: written=2000, positives=581, boxes=590, missing=0
train: written=3000, positives=757, boxes=766, missing=0
train: written=4000, positives=1115, boxes=1186, missing=0
train: written=5000, positives=1494, boxes=1641, missing=0
train: written=6000, positives=1839, boxes=2037, missing=0
train: written=7000, positives=2240, boxes=2498, missing=0
train: written=8000, positives=2478, boxes=2807, missing=0
train DONE: images=8225, positives=2554, boxes=2904, missing=0

Converting valid: 1876 rows
valid: written=1000, positives=340, boxes=514, missing=0
valid DONE: images=1876, positives=451, boxes=655, missing=0

Converting test: 3408 rows
test: written=1000, positives=484, boxes=559, missing=0
test: written=2000, positives=716, boxes=832, missing=0
test: written=3000, positives=879, boxes=1002, missing=0
test DONE: images=3408, positives=942, boxes=1068, missing=0

YOLO dataset ready: /content/varroa_

In [12]:
from pathlib import Path

def count_images(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for p in Path(folder).rglob("*") if p.suffix.lower() in exts or p.is_symlink())

def count_labels(folder):
    return sum(1 for p in Path(folder).rglob("*.txt"))

def count_positive_labels(folder):
    return sum(1 for p in Path(folder).rglob("*.txt") if p.stat().st_size > 0)

print("Train images:", count_images("/content/varroa_det/train/images"))
print("Valid images:", count_images("/content/varroa_det/valid/images"))
print("Test images:", count_images("/content/varroa_det/test/images"))

print("Train labels:", count_labels("/content/varroa_det/train/labels"))
print("Valid labels:", count_labels("/content/varroa_det/valid/labels"))
print("Test labels:", count_labels("/content/varroa_det/test/labels"))

print("Positive train labels:", count_positive_labels("/content/varroa_det/train/labels"))
print("Positive valid labels:", count_positive_labels("/content/varroa_det/valid/labels"))
print("Positive test labels:", count_positive_labels("/content/varroa_det/test/labels"))

print("\nYAML:")
print(Path("/content/varroa_det/varroa_mite.yaml").read_text())

Train images: 8223
Valid images: 1876
Test images: 3408
Train labels: 8223
Valid labels: 1876
Test labels: 3408
Positive train labels: 2553
Positive valid labels: 451
Positive test labels: 942

YAML:
path: /content/varroa_det
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: mite



In [13]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/varroa_det/varroa_mite.yaml",
    epochs=80,
    imgsz=960,
    batch=16,
    patience=20,
    project="/content/runs",
    name="varroa_mite_detector",
    exist_ok=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.4,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    close_mosaic=10,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.81 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/varroa_det/varroa_mite.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4

KeyboardInterrupt: 

In [15]:
from ultralytics import YOLO

model = YOLO("/content/runs/varroa_mite_detector/weights/best.pt")

metrics = model.val(
    data="/content/varroa_det/varroa_mite.yaml",
    split="test",
    imgsz=960,
    conf=0.25,
    iou=0.45
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 704.2±956.3 MB/s, size: 86.2 KB)
val: Scanning /content/varroa_det/test/labels... 3408 images, 2466 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3408/3408 1.1Kit/s 3.2s
val: New cache created: /content/varroa_det/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 213/213 6.3it/s 33.6s
                   all       3408       1068      0.761      0.681      0.609      0.182
Speed: 1.8ms preprocess, 5.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.6087289041813426
mAP50-95: 0.18195676570384478
Precision: 0.7612252663236888
Recall: 0.6805945832081738


In [16]:
model.predict(
    source="/content/varroa_det/test/images",
    imgsz=960,
    conf=0.25,
    save=True,
    max_det=5
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/3408 /content/varroa_det/test/images/videos__2017-09-01_10-54-26__2017-09-01_10-54-26.mp4-bee_id_7741-15-1.png_b5dfe9aa.png: 960x576 4 mites, 43.4ms
image 2/3408 /content/varroa_det/test/images/videos__2017-09-01_10-54-26__2017-09-01_10-54-26.mp4-bee_id_7742-15-1.png_e60c1435.png: 960x576 (no detections), 10.1ms
image 3/3408 /content/varroa_det/test/images/videos__2017-09-01_10-54-26__2017-09-01_10-54-26.mp4-bee_id_7743-390-1.png_ab850aba.png: 96

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'mite'}
 obb: None
 orig_img: array([[[ 84,  82,  81],
         [ 84,  82,  81],
         [ 85,  83,  82],
         ...,
         [ 99,  97,  96],
         [ 99,  97,  97],
         [100,  98,  98]],
 
        [[ 82,  80,  79],
         [ 84,  82,  81],
         [ 86,  84,  83],
         ...,
         [ 99,  97,  96],
         [ 98,  96,  97],
         [ 99,  97,  98]],
 
        [[ 82,  80,  79],
         [ 84,  82,  81],
         [ 87,  85,  84],
         ...,
         [ 99,  97,  96],
         [ 98,  96,  97],
         [ 99,  97,  98]],
 
        ...,
 
        [[ 99,  97,  96],
         [ 99,  97,  96],
         [ 99,  97,  96],
         ...,
         [100,  91,  93],
         [ 99,  90,  92],
         [ 98,  89,  91]],
 
        [[ 99,  97,  96],
         [ 99,  97,  96],
         [ 99,  97,  96],
         ...,
         [ 99,  90, 

In [17]:
from ultralytics import YOLO

model = YOLO("/content/runs/varroa_mite_detector/weights/best.pt")

model.train(
    data="/content/varroa_det/varroa_mite.yaml",
    epochs=10,
    imgsz=960,
    batch=16,
    patience=5,
    project="/content/runs",
    name="varroa_mite_detector_finetune",
    exist_ok=True,
    lr0=0.001,
    degrees=2.0,
    translate=0.04,
    scale=0.2,
    fliplr=0.5,
    mosaic=0.1,
    mixup=0.0,
    close_mosaic=3,
)

New https://pypi.org/project/ultralytics/8.4.81 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/varroa_det/varroa_mite.yaml, degrees=2.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/varroa_mite_detector/weights/best.pt, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b2d72d88fe0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [18]:
from ultralytics import YOLO

model = YOLO("/content/runs/varroa_mite_detector_finetune/weights/best.pt")

metrics = model.val(
    data="/content/varroa_det/varroa_mite.yaml",
    split="test",
    imgsz=960,
    conf=0.25,
    iou=0.45
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1456.9±441.6 MB/s, size: 85.5 KB)
val: Scanning /content/varroa_det/test/labels.cache... 3408 images, 2466 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3408/3408 840.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 213/213 6.9it/s 31.0s
                   all       3408       1068       0.88      0.716       0.68      0.229
Speed: 1.9ms preprocess, 4.8ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /content/runs/detect/val-2
mAP50: 0.6799599391332274
mAP50-95: 0.22865158495250096
Precision: 0.8803222094361335
Recall: 0.7162921348314607


In [20]:
from pathlib import Path
import shutil

src = Path("/content/runs/varroa_mite_detector_finetune/weights/best.pt")
dst = Path("/content/drive/MyDrive/CSCI435_Buzzlytics_data/weights/varroa_det.pt")

dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)

print("Saved final Varroa detector:", dst)

Saved final Varroa detector: /content/drive/MyDrive/CSCI435_Buzzlytics_data/weights/varroa_det.pt


In [19]:
!find /content/runs -path "*/weights/best.pt"

/content/runs/varroa_mite_detector/weights/best.pt
/content/runs/varroa_mite_detector_finetune/weights/best.pt
